# Imports

In [1]:
import pathlib
import pandas as pd
import numpy as np
import torch

from experiments.models import TwoStepForecast

from experiments.utils import (
    load_experiments_specs,
    create_tsfeatures,
)

# Load Experiment Specifications

In [2]:
# Load specifications
ablation_run = "A9"
train_type = "global"
dataset = "rossmann"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]

lags = meta["lags"]
max_lag = meta["max_lag"]
n_series = meta["n_series"]
series_ids = meta["series_ids"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
loss_fn = config["general"]["loss_function"]

dl_params = config["deep_learning"]
htnetar_params = config["hyper_treenet_params"]
htnetar_params_lgb = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "embedding_dimension", "use_random_projection"]}
htnetar_network_params = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "eta", "linear_tree"]}
htnetar_network_params["rp_embed_dim"] = max_lag
htnetar_network_params["hidden_dim"] = dl_params["hidden_size"]
htnetar_network_params["dropout"] = dl_params["dropout"]
htnetar_network_params["learning_rate"] = dl_params["learning_rate"]

# TS-Features

In [3]:
# Extract time series features
ts_feats_df, ts_feats = create_tsfeatures(
    train=train_df,
    freq=freq
)

# Add features to train and test
train_df = pd.merge(train_df, ts_feats_df, on="series_id", how="left")
test_df = pd.merge(test_df, ts_feats_df, on="series_id", how="left")
features = features + ts_feats

# Ablation:

Following the two-step approach described by He et al. (2014), we first train a LightGBM model (Ke et al., 2017) and extract the leaf indices from the decision trees. These leaf indices are then used as input features to a separately trained MLP.

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# Forecast with
two_step_model_fcsts = TwoStepForecast(
    htnetar_params,
    htnetar_network_params,
    train_df,
    test_df,
    features,
    lags,
    freq,
    fcst_h,
    loss_fn,
    seed,
    device
)

# Add actual values
fcsts_df = pd.merge(
    two_step_model_fcsts,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)
fcsts_df["ablation"] = ablation_run

# Output

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / "ablation" / dataset.lower()
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{ablation_run}_fcsts.csv", index=False)